<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab3-Springs-Moments-and-Equilibrium.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 3: Springs, Moments and Equilibrium

**Unit 3: Mechanical systems** · General Practical Physics I · Davidson Open University

**In this lab you will:**
- calibrate a spring and find its spring constant from a graph;
- spot the limit of proportionality in spring data;
- apply the principle of moments to balance a beam;
- calculate elastic potential energy.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '3.1': {'h': '73e2ec0b56118a64', 'hint': 'k = F / x.', 'dp': 1},
    '3.2': {'h': '888c69ff3eb04057', 'hint': 'np.polyfit(x, F, 1) returns [gradient, intercept]; take element [0].', 'ok': 'Using the gradient of all points is better than using one pair of readings.', 'dp': 1},
    '3.3': {'h': '1616ae0a2d79a77c', 'hint': 'Loop over zip(loads, ext) and stop at the first x_i > 1.10 * F_i / k_est.', 'ok': 'Beyond this load the spring is no longer linear; do not use those points to find k.'},
    '3.4': {'h': '5a3c6a5f21de5ea9', 'hint': 'm₁ d₁ = m₂ d₂, so m₂ = m₁ d₁ / d₂.', 'ok': 'Check it in Balancing Act.', 'dp': 2},
    '3.5': {'h': 'b836e0e5d7284c13', 'hint': '0.5 * k * x**2', 'dp': 3},
    '3.6': {'h': '807abb21ce640ba1', 'hint': 'return round(np.polyfit(extensions, forces, 1)[0], 1)  (x first, F second).', 'ok': 'You can now calibrate any spring from data.', 'dp': 1, 'args': [([0, 1, 2, 3, 4, 5], [0.000, 0.012, 0.025, 0.037, 0.050, 0.062]), ([0, 2, 4, 6], [0, 0.1, 0.2, 0.3]), ([1, 2, 3], [0.011, 0.019, 0.031])]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Hooke's Law](https://phet.colorado.edu/en/simulations/hookes-law)** – stretch a spring with an applied force and read the displacement.
- **[PhET Balancing Act](https://phet.colorado.edu/en/simulations/balancing-act)** – place masses on a beam and find the balance positions.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Hooke's law:** within the limit of proportionality, the force F and the extension x are proportional: **F = kx**, where k is the spring constant (N/m).

**Principle of moments:** a beam balances when the total clockwise moment equals the total anticlockwise moment. Moment = force × perpendicular distance from the pivot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
k = 50                 # N/m
for F in [1, 2, 4]:
    print(f'F = {F} N  ->  extension x = F/k = {F / k:.3f} m')

## Part B – Practise (about 20 minutes)
**B1.** A force of 4.0 N stretches a spring by 0.050 m. Find k in N/m.

In [ ]:
answer_3_1 = ...

In [ ]:
check('3.1', answer_3_1)

**B2.** Use the illustrative calibration data below. Fit a straight line of **F against x** with `np.polyfit(x, F, 1)` and take the gradient as k (1 decimal place).

In [ ]:
F = np.array([0, 1, 2, 3, 4, 5])                               # N  (illustrative)
x = np.array([0.000, 0.012, 0.025, 0.037, 0.050, 0.062])       # m  (illustrative)
plt.plot(x, F, 'o'); plt.xlabel('extension x (m)'); plt.ylabel('force F (N)'); plt.show()

In [ ]:
answer_3_2 = ...

In [ ]:
check('3.2', answer_3_2)

**B3. Limit of proportionality.** In the data below, the spring stops obeying Hooke's law near the end. Find the **first load** (in N) at which the extension is more than 10% larger than k × the earlier pattern predicts. *Hint:* the first readings give k_est = F/x; compare each predicted extension F / k_est with the measured one.

In [ ]:
loads = np.array([1, 2, 3, 4, 5, 6, 7, 8])                                  # N  (illustrative)
ext = np.array([0.020, 0.040, 0.060, 0.080, 0.100, 0.121, 0.150, 0.190])    # m
k_est = loads[0] / ext[0]

In [ ]:
answer_3_3 = ...   # YOUR CODE: the first load where ext > 1.10 * loads / k_est

In [ ]:
check('3.3', answer_3_3)

**B4. Moments.** A 2.0 kg mass sits 0.75 m to the left of a pivot. What mass (kg) must sit 0.50 m to the right to balance the beam? (The g cancels.)

In [ ]:
answer_3_4 = ...

In [ ]:
check('3.4', answer_3_4)

**B5.** Elastic potential energy is E = ½kx². Find E in joules for k = 80 N/m and x = 0.050 m (3 decimal places).

In [ ]:
answer_3_5 = ...

In [ ]:
check('3.5', answer_3_5)

## Part C – Build a tool (about 20 minutes)
Complete `spring_constant(forces, extensions)`. It should fit F against x with `np.polyfit` and return the gradient rounded to 1 decimal place.

In [ ]:
def spring_constant(forces, extensions):
    ...      # YOUR CODE

In [ ]:
check('3.6', spring_constant)

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 3 assignment.** In PhET Hooke's Law, record at least six force–displacement pairs for one spring and find k with your tool. Then, in Balancing Act, set up and record two balanced arrangements and check the principle of moments. Label the data "simulated".

In [ ]:
my_F = [...]
my_x = [...]
# print(spring_constant(my_F, my_x))

**D2. Reflect.**
1. Why does the calibration line pass (nearly) through the origin?
2. What would happen to your value of k if you used points beyond the limit of proportionality?
3. Give one everyday example of the principle of moments.

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 3 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.